# 🏥 TriMedAgent V2 - Colab/Kaggle Demo (Multi-turn Memory)

**Demo đầy đủ TriMedAgent V2 với:**
- 🔄 **Multi-turn Conversation Memory** - Hệ thống nhớ ngữ cảnh qua nhiều lượt
- 🔬 **BiomedCLIP** - Triage & Gatekeeper
- 🎯 **Grounding DINO** - Object Detection
- 🎭 **MedSAM** - Medical Segmentation
- 🧠 **LLaVA-Med** - Visual Reasoning
- 💬 **Interactive Chatbot** - Gradio Interface

---
## 1️⃣ Setup Environment

In [1]:
# @title 1.1 🖥️ Check GPU & Setup for 2x T4 (Full Power)
import torch

# --- CẤU HÌNH FULL ---
# Biến điều khiển chính (Thay thế cho biến MODE cũ)
LITE_MODE = False      # False = Chạy Full FP16 (Cần 2x T4), True = Chạy 4-bit (1x T4)
LOAD_LLAVA = True      # Bật LLaVA
LOAD_DINO = True       # Bật DINO
LOAD_SAM = True        # Bật MedSAM
LOAD_BIOMED = True     # Bật Triage
# ---------------------

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"✅ Detected {n_gpus} GPUs!")
    for i in range(n_gpus):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
    
    if n_gpus < 2 and not LITE_MODE:
        print("⚠️ Cảnh báo: Bạn đang chạy FULL FP16 trên 1 GPU. Có thể bị tràn bộ nhớ (OOM).")
        print("   Khuyên dùng: Chọn Accelerator 'GPU T4 x2' trên Kaggle hoặc đặt LITE_MODE = True.")
else:
    print("❌ Lỗi: Không tìm thấy GPU.")

KeyboardInterrupt: 

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies (TriMedAgent V2 - Final)
# @markdown Kết hợp: Clone Repo + Cài thư viện mới nhất hỗ trợ LLaVA-Med Mistral.

import os
import sys

# Kiểm tra biến môi trường
if 'LOAD_DINO' not in globals(): LOAD_DINO = True
if 'LOAD_SAM' not in globals(): LOAD_SAM = True

# 1. Clone Repo (BẮT BUỘC để có cấu trúc file)
REPO_URL = "https://github.com/ngnam1104/TriMedAgent.git"

if not os.path.exists('TriMedAgent'):
    print("📥 Cloning TriMedAgent repository...")
    !git clone {REPO_URL}
else:
    print("✅ Repository already exists.")

# Chuyển thư mục làm việc vào trong repo
%cd TriMedAgent

# 2. Install Core Dependencies (Đã thêm fix version cho LLaVA Mistral)
print("\n📦 Installing Dependencies...")

# QUAN TRỌNG: Thêm -U để update transformers/accelerate lên bản mới nhất
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q open_clip_torch==2.23.0 einops timm safetensors sentencepiece pillow requests
# Thêm nest_asyncio để fix lỗi Gradio UI
!pip install -q gradio nest_asyncio protobuf scipy

# 3. Install AI Tools (Logic cũ của bạn)
if LOAD_DINO:
    print("   + Installing GroundingDINO...")
    !pip install -q groundingdino-py
    
    # Tải file config thủ công (Tránh lỗi thiếu file config)
    print("   + Downloading GroundingDINO Config...")
    config_url = "https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py"
    os.makedirs("weights", exist_ok=True)
    !wget -q -O weights/GroundingDINO_SwinT_OGC.py {config_url}

if LOAD_SAM:
    print("   + Installing Segment Anything...")
    !pip install -q segment-anything

print("\n✅ Installation Complete.")
print("⚠️ QUAN TRỌNG: Hãy nhấn nút 'RESTART SESSION' (hoặc biểu tượng ↺) trên thanh menu ngay bây giờ!")
print("👉 Sau khi Restart, chạy tiếp từ Cell 2.1.")

In [ ]:
# @title 1.2 📦 Install & Verify Dependencies (Fix Accelerate)
# @markdown Cài đặt và kiểm tra ngay lập tức.

import sys
import subprocess

print("⏳ Installing libraries...")
# Cài đặt ép buộc bản mới
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "bitsandbytes", "protobuf", "scipy", "pillow"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio", "groundingdino-py", "segment-anything-py"])

print("✅ Installation Complete.")

# KIỂM TRA NGAY LẬP TỨC
try:
    import accelerate
    import transformers
    print(f"🎉 SUCCESS: Accelerate {accelerate.__version__} is loaded!")
    print(f"🎉 SUCCESS: Transformers {transformers.__version__} is loaded!")
except ImportError as e:
    print(f"❌ ERROR: Thư viện chưa nạp được ({e}). Vui lòng Restart Session lần nữa!")

In [ ]:
# @title 1.3 📚 Import Libraries

import sys
import os
import json
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
import base64
import requests
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
import time
import accelerate

# Add project to path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Imports successful")
print(f"   Project Root: {PROJECT_ROOT}")
print(f" Mode: {'Lite' if globals().get('LITE_MODE') else 'Full'}")

In [ ]:
# @title 1.4 ⚙️ Load Configuration

CONFIG_PATH = PROJECT_ROOT / "serve" / "labels.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, 'r') as f:
        TRIMED_CONFIG = json.load(f)
    print("✅ Loaded config from serve/labels.json")
else:
    TRIMED_CONFIG = {
        "triage_labels": [
            "Chest X-ray", "Brain MRI", "Abdominal CT", "Histopathology",
            "Ultrasound", "Dermoscopy", "Gross pathology", "Bone X-ray",
            "Lung CT", "Retinal fundus", "Mammography"
        ],
        "gatekeeper_prompts": {
            "positive": "Pathological finding, lesion, tumor, abnormality",
            "negative": "Normal tissue, healthy anatomy, background noise"
        },
        "thresholds": {
            "triage_confidence": 0.5,
            "gatekeeper_confidence": 0.6
        }
    }
    print("⚠️ Using default configuration")

print(f"\n📋 Triage Labels: {len(TRIMED_CONFIG['triage_labels'])} modalities")
print(f"📋 Gatekeeper Threshold: {TRIMED_CONFIG['thresholds']['gatekeeper_confidence']}")

In [ ]:
# @title 1.5 📥 Download Model Weights (Kaggle Optimized)
import os
import requests
from tqdm import tqdm

def download_file(url, filename):
    if os.path.exists(filename):
        print(f"✅ Found {filename}")
        return
    
    print(f"📥 Downloading {filename}...")
    try:
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        
        with open(filename, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for data in response.iter_content(chunk_size=1024):
                size = f.write(data)
                bar.update(size)
    except Exception as e:
        print(f"❌ Error downloading {filename}: {e}")

# Tạo thư mục weights
os.makedirs("weights", exist_ok=True)

# 1. Grounding DINO Weights
if globals().get('LOAD_DINO', True):
    DINO_URL = "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth"
    download_file(DINO_URL, "weights/groundingdino_swint_ogc.pth")
    
    # Config Path (Hardcoded để tránh lỗi import nếu chưa restart kernel)
    dino_config_path = os.path.join(os.getcwd(), "weights", "GroundingDINO_SwinT_OGC.py")
    print(f"📋 DINO Config Path: {dino_config_path}")

# 2. MedSAM Weights
if globals().get('LOAD_SAM', True):
    SAM_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    download_file(SAM_URL, "weights/medsam_vit_b.pth")

print("\n✅ All weights ready!")

In [ ]:

# @title 1.6 📦 Install Quantization Libraries (Required for LLaVA)
print("📦 Installing bitsandbytes for 4-bit loading...")
!pip install -q bitsandbytes
!pip install -q accelerate==0.25.0  # Đảm bảo phiên bản tương thích
print("✅ Ready for LLaVA!")

---
## 2️⃣ 🔧 Individual Tools Demo

Demo từng tool riêng biệt (không cần Orchestrator).

In [ ]:
# @title 2.1 🔬 BiomedCLIP Tool (Triage + Gatekeeper)
# @markdown Zero-shot medical image classification

import torch
import open_clip

class BiomedCLIPTool:
    """BiomedCLIP for medical image triage and verification."""
    
    def __init__(self, device="cuda"):
        self.device = device
        self.model = None
        self.preprocess = None
        self.tokenizer = None
        
    def load(self):
        """Load BiomedCLIP model."""
        print("📥 Loading BiomedCLIP...")
        
        self.model, self.preprocess, _ = open_clip.create_model_and_transforms(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224',
            device=self.device
        )
        self.tokenizer = open_clip.get_tokenizer(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.model.eval()
        print(f"✅ BiomedCLIP loaded on {self.device}")
        return self
    
    @torch.no_grad()
    def classify(self, image: Image.Image, labels: List[str]) -> Dict[str, float]:
        """Zero-shot classification."""
        if self.model is None:
            self.load()
            
        # Preprocess image
        image_input = self.preprocess(image).unsqueeze(0).to(self.device)
        
        # Tokenize labels
        text_inputs = self.tokenizer(labels).to(self.device)
        
        # Get features
        image_features = self.model.encode_image(image_input)
        text_features = self.model.encode_text(text_inputs)
        
        # Normalize
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        # Calculate similarity
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        probs = similarity[0].cpu().numpy()
        
        return {label: float(prob) for label, prob in zip(labels, probs)}
    
    def triage(self, image: Image.Image) -> tuple:
        """Determine medical image modality."""
        labels = TRIMED_CONFIG["triage_labels"]
        scores = self.classify(image, labels)
        
        best_label = max(scores, key=scores.get)
        best_score = scores[best_label]
        
        return best_label, best_score, scores
    
    def verify_region(self, image: Image.Image, target_label: str) -> tuple:
        """Verify if a region contains pathology (Gatekeeper)."""
        pos_label = f"{TRIMED_CONFIG['gatekeeper_prompts']['positive']} of {target_label}"
        neg_label = TRIMED_CONFIG['gatekeeper_prompts']['negative']
        
        scores = self.classify(image, [pos_label, neg_label])
        
        pathology_score = scores[pos_label]
        normal_score = scores[neg_label]
        threshold = TRIMED_CONFIG['thresholds']['gatekeeper_confidence']
        
        is_valid = pathology_score > threshold
        
        return is_valid, pathology_score, normal_score

# Initialize (don't load yet)
biomedclip_tool = None

if LOAD_BIOMED:  # Thay vì check MODE != "demo"
    # Nếu có 2 GPU, để BiomedCLIP ở GPU 0 chung với LLaVA (nó nhẹ)
    # Tìm chỗ khởi tạo BiomedCLIPTool và sửa:
    device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0" # Đẩy sang GPU 1
    biomedclip_tool = BiomedCLIPTool(device=device)
    print("🔬 BiomedCLIP tool ready (call .load() to initialize)")
else:
    print("⚠️ BiomedCLIP disabled")

In [ ]:
# @title 2.2 🎯 Grounding DINO Tool (Fixed: Output as List)
import os
import torch
from groundingdino.util.inference import load_model, load_image, predict, annotate

class GroundingDINOTool:
    def __init__(self, device=None):
        # Tự động chọn GPU 1 nếu có 2 GPU, ngược lại dùng GPU 0
        if device is None:
            self.device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
        else:
            self.device = device
            
        self.config_path = "weights/GroundingDINO_SwinT_OGC.py"
        self.checkpoint_path = "weights/groundingdino_swint_ogc.pth"
        self.model = None

    def load(self):
        if self.model is None:
            print(f"🎯 Loading Grounding DINO on {self.device}...")
            self.model = load_model(self.config_path, self.checkpoint_path, device=self.device)
        return self

    def detect(self, image, text_prompt, box_threshold=0.45, text_threshold=0.25):
        if self.model is None: self.load()
        
        # Transform ảnh chuẩn của DINO
        import groundingdino.datasets.transforms as T
        from PIL import Image
        import numpy as np

        transform = T.Compose([
            T.RandomResize([800], max_size=1333),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        image_source, _ = transform(image, None)
        
        # Dự đoán
        boxes, logits, phrases = predict(
            model=self.model,
            image=image_source,
            caption=text_prompt,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            device=self.device
        )
        
        # 1. Chuyển đổi Boxes sang XYXY (List float)
        w, h = image.size
        boxes_xyxy = []
        for box in boxes:
            boxes_xyxy.append([
                float(box[0] * w - box[2] * w / 2),
                float(box[1] * h - box[3] * h / 2),
                float(box[0] * w + box[2] * w / 2),
                float(box[1] * h + box[3] * h / 2),
            ])
            
        # 2. FIX QUAN TRỌNG: Chuyển Scores từ Tensor sang List Python
        # Điều này giúp np.argsort() hoạt động không bị lỗi
        scores_list = logits.cpu().tolist()
            
        return {"boxes": boxes_xyxy, "scores": scores_list, "labels": phrases}

# Khởi tạo lại tool
dino_tool = None
if globals().get('LOAD_DINO', True):
    dino_tool = GroundingDINOTool()
    print(f"🎯 DINO Tool Defined (Target: {dino_tool.device})")

In [ ]:
# @title 2.3 🎭 MedSAM Tool (Force GPU 1)
import torch
from segment_anything import sam_model_registry, SamPredictor
import numpy as np

class MedSAMTool:
    def __init__(self, device=None):
        # Ép sang GPU 1
        if device is None:
            self.device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
        else:
            self.device = device
            
        self.checkpoint = "weights/medsam_vit_b.pth"
        self.model_type = "vit_b"
        self.medsam_model = None
        self.predictor = None

    def load(self):
        if self.medsam_model is None:
            print(f"🎭 Loading MedSAM on {self.device}...")
            self.medsam_model = sam_model_registry[self.model_type](checkpoint=self.checkpoint)
            self.medsam_model.to(device=self.device)
            self.predictor = SamPredictor(self.medsam_model)
        return self

    def segment(self, image, boxes):
        if self.predictor is None: self.load()
        
        image_np = np.array(image)
        self.predictor.set_image(image_np)
        
        generated_masks = []
        for box in boxes:
            box_np = np.array(box)
            masks, _, _ = self.predictor.predict(
                point_coords=None,
                point_labels=None,
                box=box_np[None, :],
                multimask_output=False,
            )
            generated_masks.append(masks[0])
            
        return generated_masks

medsam_tool = None
if globals().get('LOAD_SAM', True):
    medsam_tool = MedSAMTool()
    print(f"🎭 MedSAM Tool Defined (Target: {medsam_tool.device})")

In [ ]:
# @title 2.4 🧪 Test Individual Tools
# @markdown Chạy test với ảnh mẫu

# Download sample image
SAMPLE_URL = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQwzZYDXuuzIBdCZScPmBJOGCoJXqmi73NtwQ&s"

print("📥 Downloading sample image...")
response = requests.get(SAMPLE_URL)
sample_image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"✅ Sample image: {sample_image.size}")

# Display
plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Medical Image")
plt.axis('off')
plt.show()

In [ ]:
# @title 2.4.1 Test BiomedCLIP (Triage)

if biomedclip_tool is not None:
    print("🔬 Loading BiomedCLIP...")
    biomedclip_tool.load()
    
    print("\n🔬 Running Triage...")
    modality, confidence, all_scores = biomedclip_tool.triage(sample_image)
    
    print(f"\n📊 Triage Result:")
    print(f"   Modality: {modality}")
    print(f"   Confidence: {confidence:.1%}")
    
    print(f"\n   Top 5 Scores:")
    sorted_scores = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)
    for label, score in sorted_scores[:5]:
        bar = "█" * int(score * 30)
        print(f"   {label}: {score:.1%} {bar}")
else:
    # Demo mode
    print("🔬 [DEMO] Triage Result:")
    print("   Modality: Chest X-ray")
    print("   Confidence: 92.5%")

In [ ]:
# @title 2.4.2 Test Grounding DINO (Detection) - CLEAN VISUALIZATION

if dino_tool is not None:
    print("🎯 Loading Grounding DINO...")
    dino_tool.load()
    
    print("\n🎯 Running Detection (with NMS)...")
    detection_query = "lung nodule" 
    # Bạn có thể đổi query khác nếu muốn: "rib fracture", "pneumonia", "tumor"
    
    detection_result = dino_tool.detect(sample_image, detection_query)
    
    print(f"\n📊 Detection Result:")
    print(f"   Query: '{detection_query}'")
    print(f"   Boxes Found: {len(detection_result['boxes'])}")
    
    # --- VISUALIZATION MỚI: SẠCH VÀ ĐẸP ---
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(sample_image)
    
    boxes = detection_result['boxes']
    scores = detection_result['scores']   # <--- ĐÚNG
    labels = detection_result['labels']   # <--- ĐÚNG
    # Chỉ vẽ tối đa 5 box có điểm cao nhất
    if len(scores) > 0:
        # Sắp xếp giảm dần theo điểm
        sorted_indices = np.argsort(scores)[::-1][:5]
        
        for i in sorted_indices:
            x1, y1, x2, y2 = boxes[i]
            score = scores[i]
            label = labels[i]
            
            # Vẽ Box: Nét mảnh, màu đỏ tươi
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='#FF3333', facecolor='none'
            )
            ax.add_patch(rect)
            
            # Vẽ Label: Nền đỏ, chữ trắng, font nhỏ
            text = f"{label}: {score:.2f}"
            ax.text(
                x1, y1-8, text, 
                color='white', fontsize=9, weight='bold',
                bbox=dict(facecolor='#FF3333', alpha=0.7, pad=2, edgecolor='none')
            )
    else:
        print("⚠️ No objects detected.")

    ax.set_title(f"Detection: '{detection_query}' (Top 5)")
    ax.axis('off')
    plt.show()
else:
    print("🎯 [DEMO] Detection Result:")
    print("   Query: 'lung nodule'")
    print("   Boxes Found: 1")
    print("   Box: [88, 132, 264, 308], Score: 0.75")

In [ ]:
# @title 2.4.3 Test MedSAM (Segmentation)
# @markdown Test khả năng phân vùng dựa trên Box giả định hoặc Box từ DINO.

if medsam_tool is not None:
    print("🎭 Loading MedSAM...")
    medsam_tool.load()
    
    # Lấy box từ kết quả DINO ở trên (nếu có), nếu không thì dùng box mẫu
    if 'detection_result' in globals() and len(detection_result['boxes']) > 0:
        test_boxes = detection_result['boxes']
        print(f"🔗 Using {len(test_boxes)} boxes from Grounding DINO result.")
    else:
        # Box mẫu (Vùng phổi phải)
        w, h = sample_image.size
        test_boxes = [[0.2*w, 0.3*h, 0.6*w, 0.7*h]]
        print("⚠️ No DINO boxes found, using dummy box.")

    print("\n🎭 Running Segmentation...")
    masks = medsam_tool.segment(sample_image, test_boxes)
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(sample_image)
    
    # Vẽ nền Mask màu đỏ bán trong suốt
    overlay = np.zeros((*sample_image.size[::-1], 4))
    for i, mask in enumerate(masks):
        overlay[mask > 0] = [1, 0, 0, 0.4] # Red, Alpha=0.4
        
        # Vẽ viền box
        x1, y1, x2, y2 = test_boxes[i]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='yellow', facecolor='none', linestyle='--')
        ax.add_patch(rect)
        
    ax.imshow(overlay)
    ax.set_title(f"MedSAM Segmentation ({len(masks)} regions)")
    ax.axis('off')
    plt.show()
else:
    print("⚠️ MedSAM not loaded (Check MODE='full')")

In [ ]:
# @title 2.5 🧠 LLaVA-Med Tool (Force Vietnamese)
# @markdown ✅ Đã thêm cơ chế **Hard Prompting** để ép model trả lời Tiếng Việt 100%.

import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

class LLaVATool:
    def __init__(self, load_in_4bit=True):
        self.model_path = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
        self.model = None
        self.processor = None
        self.device = "cuda:0" 

    def load(self):
        print(f"🧠 Loading LLaVA-Med from {self.model_path}...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        try:
            self.processor = AutoProcessor.from_pretrained(self.model_path)
            self.model = LlavaForConditionalGeneration.from_pretrained(
                self.model_path,
                quantization_config=bnb_config,
                torch_dtype=torch.float16,
                device_map={"": 0}, 
                low_cpu_mem_usage=True
            )
            print("✅ LLaVA-Med (Vietnamese Forced) Loaded!")
        except Exception as e:
            print(f"❌ Error loading LLaVA: {e}")
        return self

    def chat(self, image, query, system_context=""):
        if self.model is None: self.load()
        if self.model is None: return "❌ Lỗi: Model chưa tải."

        # --- CHIẾN THUẬT HARD PROMPT ---
        # Chèn chỉ thị bắt buộc vào ngay sát câu hỏi
        force_vn_prompt = (
            "YÊU CẦU: Đóng vai bác sĩ chẩn đoán hình ảnh người Việt Nam. "
            "Phân tích ảnh và trả lời câu hỏi bên dưới hoàn toàn bằng Tiếng Việt. "
            "Dùng từ ngữ chuyên môn y khoa Tiếng Việt chính xác.\n"
            "----------------\n"
        )
        
        # Kết hợp: [System] + [Hard Prompt] + [User Question]
        final_query = f"{system_context}\n{force_vn_prompt}\nCâu hỏi: {query}"
        
        # Format chuẩn LLaVA
        prompt = f"USER: <image>\n{final_query}\nASSISTANT:"

        inputs = self.processor(text=prompt, images=image, return_tensors="pt")
        
        # Đẩy sang GPU
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(self.device, dtype=torch.float16)
        if "input_ids" in inputs:
            inputs["input_ids"] = inputs["input_ids"].to(self.device)
        if "attention_mask" in inputs:
            inputs["attention_mask"] = inputs["attention_mask"].to(self.device)

        try:
            with torch.inference_mode():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=True,
                    temperature=0.2, # Giữ thấp để model tuân thủ lệnh
                    top_p=0.9
                )
            output = self.processor.batch_decode(output_ids, skip_special_tokens=True)[0]
            if "ASSISTANT:" in output:
                output = output.split("ASSISTANT:")[-1].strip()
            return output
        except Exception as e:
            return f"Lỗi xử lý: {str(e)}"

# Init
llava_tool = None
if 'LOAD_LLAVA' in globals() and LOAD_LLAVA:
    llava_tool = LLaVATool(load_in_4bit=True)
    print("🧠 LLaVA Tool Defined")

---
## 3️⃣ 🎯 Orchestrator Demo

Demo **Thin Client Orchestrator** - điều phối pipeline tự động.

In [ ]:
# @title 3.1 📦 Local Orchestrator V8 (Ultimate Agent: Smart Zoom & Fallback)
# @markdown ✅ **Tính năng:** Tự động dịch prompt + Zoom thông minh + Tự động tìm lại (Fallback) nếu thất bại.

import torch
import time
import re
import numpy as np
from PIL import Image
from dataclasses import dataclass, field
from typing import List, Any

# --- CONFIG CẤU HÌNH ---
ZOOM_CONFIDENCE = 0.15      # Ngưỡng thấp khi đã Zoom (vì đã loại bớt nhiễu)
GLOBAL_CONFIDENCE = 0.25    # Ngưỡng cao khi tìm toàn ảnh
STRICT_MAX_RATIO = 0.20     # Nodule không được lớn hơn 20% ảnh

@dataclass
class PipelineResult:
    llava_response: str = ""
    verified_boxes: List = field(default_factory=list)
    masks: List = field(default_factory=list)
    execution_time: float = 0.0
    execution_log: List[str] = field(default_factory=list)
    mode_used: str = ""

class TriMedOrchestratorLocal:
    def __init__(self, biomedclip=None, llava=None, dino=None, medsam=None):
        self.biomedclip = biomedclip
        self.llava = llava
        self.dino = dino
        self.medsam = medsam
        print(f"🎯 Orchestrator V8 (Ultimate) Ready")

    def log(self, result, message):
        print(message)
        result.execution_log.append(message)

    # 1. VISUAL TRANSLATION: Biến query y khoa thành query hình học
    def _get_visual_prompt(self, user_query):
        query = user_query.lower()
        mapping = {
            "nodule": "small round white spot",
            "mass": "large white area",
            "pneumonia": "cloudy white opacity",
            "fracture": "bone break gap",
            "cardiomegaly": "enlarged heart silhouette",
            "pleural effusion": "white fluid triangle base",
            "u phổi": "round white spot",
            "nốt": "small white dot"
        }
        # Tìm keyword dài nhất khớp với query
        for k, v in mapping.items():
            if k in query:
                return v
        return "abnormality" # Mặc định

    # 2. SMART PARSING: Hiểu vị trí LLaVA nói
    def _get_crop_region(self, image, hint_text):
        W, H = image.size
        hint = hint_text.lower()
        
        # Check từ khóa
        has_right = "right" in hint   # Phổi Phải (Ảnh Trái)
        has_left = "left" in hint     # Phổi Trái (Ảnh Phải)
        has_upper = any(k in hint for k in ["upper", "top", "apex"])
        has_lower = any(k in hint for k in ["lower", "bottom", "base"])

        # Logic xác định vùng
        x0, y0, x1, y1 = 0, 0, W, H
        region_name = "Full Image"
        
        # Xử lý trục Ngang (Lưu ý ngược bên: Phổi Phải -> Ảnh Trái)
        if has_right and not has_left:
            x1 = W // 2 + 50 # Lấy dư ra một chút ở giữa
            region_name = "Right Lung (Image Left)"
        elif has_left and not has_right:
            x0 = W // 2 - 50
            region_name = "Left Lung (Image Right)"
            
        # Xử lý trục Dọc
        if has_upper:
            y1 = H // 2 + 50
            region_name += " - Upper"
        elif has_lower:
            y0 = H // 2 - 50
            region_name += " - Lower"

        return (x0, y0, x1, y1), region_name

    def run_full_chain(self, image, query, **kwargs):
        start_time = time.time()
        result = PipelineResult()
        W, H = image.size
        
        # --- BƯỚC 1: CHUẨN BỊ CHIẾN THUẬT ---
        visual_prompt = self._get_visual_prompt(query)
        self.log(result, f"🔧 [SETUP] Query: '{query}' -> Visual Prompt: '{visual_prompt}'")
        
        # --- BƯỚC 2: HỎI LLaVA VỊ TRÍ (BRAIN) ---
        self.log(result, "🧠 [STEP 1] Asking LLaVA for location...")
        loc_prompt = "Where is the abnormality? Answer specific: Right/Left Lung, Upper/Lower Lobe."
        
        location_hint = "unknown"
        if self.llava:
            try:
                location_hint = self.llava.chat(image, loc_prompt)
                self.log(result, f"   🗣️ LLaVA Thought: '{location_hint}'")
            except: pass
            
        # --- BƯỚC 3: THỰC HIỆN ZOOM SEARCH (EYES) ---
        crop_box, region_name = self._get_crop_region(image, location_hint)
        self.log(result, f"   ✂️ [STEP 2] Zooming into: {region_name}")
        
        cropped_img = image.crop(crop_box)
        final_boxes = []
        
        if self.dino:
            # Chạy DINO trên vùng Crop
            det = self.dino.detect(cropped_img, visual_prompt, box_threshold=ZOOM_CONFIDENCE)
            raw_boxes = det.get('boxes', [])
            
            # Map tọa độ & Lọc
            ox, oy = crop_box[0], crop_box[1]
            for box in raw_boxes:
                # Chuyển tọa độ về ảnh gốc
                real_x1, real_y1 = box[0] + ox, box[1] + oy
                real_x2, real_y2 = box[2] + ox, box[3] + oy
                
                # Tính lại Ratio so với ẢNH GỐC
                area = (real_x2 - real_x1) * (real_y2 - real_y1)
                ratio = area / (W * H)
                
                if 0.0005 < ratio < STRICT_MAX_RATIO:
                    final_boxes.append([real_x1, real_y1, real_x2, real_y2])
                    self.log(result, f"      ✅ Found in Zoom: Ratio={ratio:.4f}")
        
        result.mode_used = "Zoom Search"

        # --- BƯỚC 4: FALLBACK (NẾU ZOOM THẤT BẠI) ---
        if not final_boxes and self.dino:
            self.log(result, "   ⚠️ [FALLBACK] Zoom failed. Trying Global Search...")
            det_global = self.dino.detect(image, visual_prompt, box_threshold=GLOBAL_CONFIDENCE)
            raw_global = det_global.get('boxes', [])
            
            for box in raw_global:
                area = (box[2] - box[0]) * (box[3] - box[1])
                ratio = area / (W * H)
                if 0.0005 < ratio < STRICT_MAX_RATIO:
                    final_boxes.append(box)
                    self.log(result, f"      ✅ Found in Global: Ratio={ratio:.4f}")
            result.mode_used = "Global Fallback"

        # --- BƯỚC 5: KẾT QUẢ & MEDSAM ---
        result.verified_boxes = final_boxes
        
        if final_boxes and self.medsam:
            self.log(result, f"🎭 [STEP 3] Running MedSAM on {len(final_boxes)} regions...")
            result.masks = self.medsam.segment(image, final_boxes)
        else:
            self.log(result, "❌ [RESULT] No valid regions found.")

        # --- BƯỚC 6: TRẢ LỜI USER ---
        if self.llava:
            context = (
                f"System Report: Detect {len(final_boxes)} '{visual_prompt}' regions. "
                f"Location hint used: {location_hint}. Mode: {result.mode_used}."
            )
            if not final_boxes:
                context = "System Report: No abnormality detected by visual tools."
                
            result.llava_response = self.llava.chat(image, query, system_context=context)

        result.execution_time = time.time() - start_time
        return result

# Khởi tạo lại
orchestrator = TriMedOrchestratorLocal(
    biomedclip=biomedclip_tool if 'biomedclip_tool' in globals() else None,
    llava=llava_tool if 'llava_tool' in globals() else None,
    dino=dino_tool if 'dino_tool' in globals() else None,
    medsam=medsam_tool if 'medsam_tool' in globals() else None
)

In [ ]:
# @title 3.2 🚀 PRE-LOAD ALL MODELS (V2 Compatible)
# @markdown Nạp cưỡng bức toàn bộ model vào VRAM (Tương thích cả V1 và V2)

import torch
import time

def print_gpu_status():
    print(f"\n📊 GPU Memory Status:")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            mem = torch.cuda.memory_allocated(i) / 1024**3
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"   - GPU {i}: {mem:.2f} GB / {total:.1f} GB")
    else:
        print("   - No GPU detected.")

print("📉 Trạng thái VRAM hiện tại (Trước khi nạp):")
print_gpu_status()

print("\n========================================================")
print("🚀 ĐANG NẠP TOÀN BỘ MODEL (FULL PRE-LOAD)...")
print("========================================================")

if 'orchestrator' in globals():
    orc = globals()['orchestrator']
    
    # --- 1. NẠP LLaVA (Nặng nhất) ---
    # Tự động tìm biến 'llava' (V2) hoặc 'vlm_model' (V1)
    llava_instance = getattr(orc, 'llava', getattr(orc, 'vlm_model', None))
    
    if llava_instance:
        print("\n🧠 1. Đang nạp LLaVA (Visual Reasoning)...")
        start_time = time.time()
        # Gọi hàm load()
        if hasattr(llava_instance, 'load'):
            llava_instance.load()
        print(f"   ✅ LLaVA đã sẵn sàng! (Mất {time.time() - start_time:.1f}s)")
    else:
        print("   ⚠️ Không tìm thấy LLaVA (Skipped).")

    # --- 2. NẠP DINO ---
    dino_instance = getattr(orc, 'dino', getattr(orc, 'detector_model', None))
    if dino_instance:
        print("\n🎯 2. Kiểm tra Grounding DINO...")
        if hasattr(dino_instance, 'model'):
            dino_instance.model.to(dino_instance.device)
        print("   ✅ DINO đã sẵn sàng!")

    # --- 3. NẠP MedSAM ---
    medsam_instance = getattr(orc, 'medsam', getattr(orc, 'segmentor_model', None))
    if medsam_instance:
        print("\n🎭 3. Kiểm tra MedSAM...")
        if hasattr(medsam_instance, 'medsam_model'):
            medsam_instance.medsam_model.to(medsam_instance.device)
        print("   ✅ MedSAM đã sẵn sàng!")

    # --- 4. NẠP BiomedCLIP ---
    biomed_instance = getattr(orc, 'biomedclip', getattr(orc, 'triage_model', None))
    if biomed_instance:
        print("\n🔬 4. Kiểm tra BiomedCLIP...")
        if hasattr(biomed_instance, 'model'):
            biomed_instance.model.to(biomed_instance.device)
        print("   ✅ BiomedCLIP đã sẵn sàng!")

else:
    print("❌ LỖI: Orchestrator chưa được khởi tạo. Hãy chạy Cell 3.0 (hoặc 3.1) chứa class TriMedOrchestratorLocal trước!")

print("\n========================================================")
print("📈 Trạng thái VRAM sau khi nạp (Kỳ vọng ~20GB tổng):")
print_gpu_status()
print("========================================================")
print("✅ HỆ THỐNG ĐÃ SẴN SÀNG! BẠN CÓ THỂ CHẠY CELL GRADIO WEB UI NGAY.")

In [ ]:
# @title 3.2 🚀 Initialize Orchestrator (Fixed MedSAM Attribute)
# @markdown Khởi tạo Orchestrator và nạp các tools (nếu chưa nạp).

import torch

print("🚀 Initializing Orchestrator...")

# 1. Load BiomedCLIP (nếu chưa load)
if 'biomedclip_tool' in globals() and biomedclip_tool:
    if not hasattr(biomedclip_tool, 'model') or biomedclip_tool.model is None:
        print("   - Loading BiomedCLIP...")
        biomedclip_tool.load()

# 2. Load Grounding DINO (nếu chưa load)
if 'dino_tool' in globals() and dino_tool:
    if not hasattr(dino_tool, 'model') or dino_tool.model is None:
        print("   - Loading Grounding DINO...")
        dino_tool.load()

# 3. Load MedSAM (FIX LỖI Ở ĐÂY)
if 'medsam_tool' in globals() and medsam_tool:
    # Sửa .model thành .medsam_model
    if not hasattr(medsam_tool, 'medsam_model') or medsam_tool.medsam_model is None:
        print("   - Loading MedSAM...")
        medsam_tool.load()

# 4. Khởi tạo Orchestrator
# Đảm bảo biến toàn cục tồn tại
if 'llava_tool' not in globals(): llava_tool = None
if 'biomedclip_tool' not in globals(): biomedclip_tool = None
if 'dino_tool' not in globals(): dino_tool = None
if 'medsam_tool' not in globals(): medsam_tool = None

orchestrator = TriMedOrchestratorLocal(
    biomedclip=biomedclip_tool,
    llava=llava_tool,
    dino=dino_tool,
    medsam=medsam_tool,
)


In [ ]:
# # @title 3.3 ▶️ Run Full Pipeline

# # Define query
# user_query = "Find and segment any abnormalities in this image"

# print(f"🎯 Query: '{user_query}'")
# print("=" * 60)

# # Run pipeline
# result = orchestrator.run_full_chain(
#     image=sample_image,
#     user_query=user_query,
#     skip_gatekeeper=False,
#     skip_segmentation=(not globals().get('LOAD_SAM', False))
# )

# print("\n" + "=" * 60)
# print("📋 PIPELINE SUMMARY")
# print("=" * 60)
# print(f"   Triage: {result.triage.modality} ({result.triage.confidence:.1%})")
# print(f"   Raw Boxes: {len(result.dino_raw_boxes)}")
# print(f"   Verified Boxes: {len(result.verified_boxes)}")
# print(f"   Masks: {len(result.masks)}")
# print(f"   Time: {result.execution_time:.2f}s")
# print(f"   Complete: {result.pipeline_complete}")

In [ ]:
# @title 3.3 ▶️ Run Full Pipeline (Smart Retry Demo & Visualization)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# --- 1. Hàm hỗ trợ vẽ (Helper Functions) ---
def show_mask(mask, ax):
    color = np.array([30/255, 144/255, 255/255, 0.6]) # Màu xanh + độ trong suốt 0.6
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    # Vẽ hình chữ nhật đỏ
    rect = patches.Rectangle((x0, y0), w, h, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)

# --- 2. Setup Query ---
# Ví dụ: "Find lung nodule" (Dễ) vs "Find abnormality" (Khó hơn)
user_query = "Please find the lung nodule"

print(f"👤 User Query: '{user_query}'")
print("-" * 60)

# --- 3. Chạy Pipeline ---
result = orchestrator.run_full_chain(
    image=sample_image,
    query=user_query
)

print("-" * 60)
print("📊 FINAL SUMMARY:")
modality = result.triage.modality if (hasattr(result, 'triage') and result.triage) else 'N/A'
print(f"   • Modality: {modality}")
print(f"   • Verified Regions: {len(result.verified_boxes)}")
print(f"   • Masks Generated: {len(result.masks)}")
print(f"   • Time: {result.execution_time:.2f}s")

# --- 4. Hiển thị Text Response ---
print("\n📝 LLaVA Response:")
print(result.llava_response)

# --- 5. VISUALIZATION (Vẽ ảnh) ---
print("\n📸 [VISUALIZE] Hiển thị kết quả...")

plt.figure(figsize=(10, 10))
plt.imshow(sample_image) # Hiển thị ảnh gốc

# Vẽ từng mask tìm được
if result.masks:
    for mask in result.masks:
        show_mask(mask, plt.gca())

# Vẽ từng box tìm được
if result.verified_boxes:
    for box in result.verified_boxes:
        show_box(box, plt.gca())

plt.axis('off')
plt.title(f"Detected: {len(result.masks)} regions")
plt.show()

In [ ]:
# @title 3.4 📊 Visualize Pipeline Results - FIXED
# @markdown Visualization rõ ràng hơn với Mask Overlay bán trong suốt.

def visualize_results(image, result):
    """Visualize pipeline results clearly."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    
    img_array = np.array(image)
    
    # --- PANEL 1: Input & Triage ---
    axes[0].imshow(img_array)
    title = "Input Image"
    if result.triage:
        title += f"\n{result.triage.modality} ({result.triage.confidence:.1%})"
    axes[0].set_title(title, fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # --- PANEL 2: Detection (Verified Boxes) ---
    axes[1].imshow(img_array)
    # Vẽ Box bị loại (mờ, nét đứt)
    for box in result.dino_raw_boxes:
        if box not in result.verified_boxes:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='gray', facecolor='none', linestyle='--', alpha=0.5)
            axes[1].add_patch(rect)
            
    # Vẽ Verified Box (Đậm, Xanh)
    for box in result.verified_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=3, edgecolor='#00FF00', facecolor='none')
        axes[1].add_patch(rect)
        
    axes[1].set_title(f"Detection + Gatekeeper\nVerified: {len(result.verified_boxes)} / {len(result.dino_raw_boxes)}", fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # --- PANEL 3: Segmentation (Mask Overlay) ---
    # Copy ảnh gốc để vẽ đè
    masked_image = img_array.copy()
    
    if result.masks and len(result.masks) > 0:
        # Tạo overlay màu
        overlay = np.zeros_like(img_array)
        
        for mask in result.masks:
            # Mask là binary (0 hoặc 255), chuyển thành bool
            mask_bool = mask > 0
            # Tô màu đỏ (Red) vào vùng mask
            overlay[mask_bool] = [255, 0, 0] 
            
        # Blend ảnh gốc và mask (alpha = 0.4)
        masked_image = 0.6 * masked_image + 0.4 * overlay
        masked_image = masked_image.astype(np.uint8)
        
    axes[2].imshow(masked_image)
    
    # Vẽ lại box lên ảnh segment cho dễ nhìn
    for box in result.verified_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='white', facecolor='none', linestyle=':')
        axes[2].add_patch(rect)

    axes[2].set_title(f"Segmentation\n{len(result.masks)} Regions", fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Chạy visualize với kết quả hiện tại
if 'result' in globals() and 'sample_image' in globals():
    visualize_results(sample_image, result)

---
## 4️⃣ 💬 Interactive Multi-turn Chatbot

Chatbot với **Gradio** hỗ trợ multi-turn conversation memory.

### ⚠️ Yêu cầu trước khi chạy:
1. Cài đặt `nest_asyncio` để tránh lỗi asyncio trên Colab
2. Chạy cell cài đặt dependencies bên dưới **trước tiên**

In [ ]:
# @title 4.0.1 📦 Install Gradio Dependencies (Kaggle/Colab Fix)
# @markdown **QUAN TRỌNG: Chạy cell này TRƯỚC khi chạy Gradio UI**

import subprocess
import sys
import os

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules
ENV_NAME = "Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"

print(f"🔍 Detected environment: {ENV_NAME}")
print("📦 Installing/Updating dependencies...")

# Install packages
packages = [
    "nest_asyncio",
    "gradio>=4.0.0,<5.0.0",  # Gradio 4.x stable
]

# Kaggle fix: downgrade uvicorn để tránh loop_factory issue
if IS_KAGGLE:
    packages.append("uvicorn<0.30.0")  # Version cũ không có loop_factory
    print("⚠️ Kaggle detected: Downgrading uvicorn to avoid asyncio issues")

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Apply nest_asyncio
import nest_asyncio
nest_asyncio.apply()

print(f"\n✅ Dependencies installed!")
print(f"✅ nest_asyncio applied!")

# Verify versions
import gradio as gr
print(f"📦 Gradio version: {gr.__version__}")

try:
    import uvicorn
    print(f"📦 uvicorn version: {uvicorn.__version__}")
except:
    pass

print("\n🎉 Ready to run Gradio UI!")

In [ ]:
# @title 4.1 🤖 Multi-turn Medical Chatbot Interface

import gradio as gr

# Global reference
current_result = None

def process_image(image):
    """Process uploaded image and run triage."""
    global orchestrator
    
    if image is None:
        return "⚠️ Please upload an image first."
    
    # Set image in orchestrator (triggers triage on first chat)
    orchestrator.set_image(image)
    orchestrator._current_image = image
    
    # Run triage preview
    triage_info = orchestrator.triage(image)
    modality, conf, _ = triage_info if triage_info else ("Unknown", 0, {})
    
    return f"""✅ **Image Loaded**
    
📊 **Triage Result:**
- **Modality**: {modality}
- **Confidence**: {conf:.1%}

📝 **Session Info:**
- **Session ID**: {orchestrator.state.session_id}
- **Chat History**: {len(orchestrator.state.messages)} messages

💡 **Try asking:**
- "What can you see in this image?"
- "Are there any abnormalities?"
- "Find and segment any tumors"
- "What is the diagnosis?"
"""

def chat(message, history):
    """Multi-turn chat with the medical AI."""
    global orchestrator, current_result
    
    if not message.strip():
        return history
    
    # Check if image is loaded
    if not hasattr(orchestrator, '_current_image') or orchestrator._current_image is None:
        history.append([message, "⚠️ Please upload a medical image first."])
        return history
    
    # Chat with orchestrator (handles multi-turn context)
    response, result = orchestrator.chat(message)
    current_result = result
    
    # Add turn info
    turn_info = f"\n\n---\n📊 *Turn {orchestrator.state.turn_count} | Time: {result.execution_time:.2f}s*"
    full_response = response + turn_info
    
    history.append([message, full_response])
    return history

def get_visualization():
    """Get current visualization with boxes and masks."""
    global orchestrator, current_result
    
    if not hasattr(orchestrator, '_current_image') or orchestrator._current_image is None:
        return None
    
    image = orchestrator._current_image
    
    if current_result is None or not current_result.verified_boxes:
        return image
    
    # Create visualization
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(np.array(image))
    
    # Draw verified boxes
    for i, box in enumerate(current_result.verified_boxes):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=3, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-10, f"Region {i+1}", color='lime', fontsize=12, fontweight='bold')
    
    # Overlay masks
    if current_result.masks:
        for mask in current_result.masks:
            if isinstance(mask, np.ndarray):
                overlay = np.zeros((*mask.shape[:2], 4))
                overlay[mask > 0] = [1, 0, 0, 0.4]
                ax.imshow(overlay)
    
    ax.axis('off')
    ax.set_title(f"Detection: {len(current_result.verified_boxes)} verified regions")
    
    # Convert to image
    fig.canvas.draw()
    img_array = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img_array = img_array.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    
    return Image.fromarray(img_array)

def clear_all():
    """Clear chat and reset session."""
    global orchestrator, current_result
    orchestrator.clear_session()
    orchestrator._current_image = None
    current_result = None
    return [], None, None, "🗑️ Session cleared. Upload a new image to start."

def show_history():
    """Show conversation history summary."""
    global orchestrator
    if not orchestrator.state or len(orchestrator.state.messages) == 0:
        return "No conversation history."
    
    summary = f"**Session: {orchestrator.state.session_id}**\n"
    summary += f"**Total turns: {orchestrator.state.turn_count}**\n\n"
    
    for i, msg in enumerate(orchestrator.state.messages[-10:]):  # Last 10 messages
        role_icon = "👤" if msg.role == "user" else "🤖"
        summary += f"{role_icon} **{msg.role.upper()}**: {msg.content[:100]}...\n\n"
    
    return summary

print("✅ Chatbot interface defined")
print("   Run next cell to launch")

In [ ]:
# @title 🩹 4.1.2 ULTIMATE FIX for Gradio Schema (Mạnh hơn)
# @markdown Chạy cell này để sửa triệt để lỗi "Cannot parse schema True/False"

import gradio_client.utils
from gradio_client.utils import APIInfoParseError

# 1. Lưu lại hàm gốc (nếu chưa lưu)
if not hasattr(gradio_client.utils, "_original_json_schema_to_python_type"):
    gradio_client.utils._original_json_schema_to_python_type = gradio_client.utils._json_schema_to_python_type

# 2. Định nghĩa hàm vá lỗi mới (Bao bọc toàn diện)
def patched_json_schema_to_python_type(schema, defs=None):
    # Nếu schema là boolean (nguyên nhân gây lỗi), trả về "Any" ngay lập tức
    if isinstance(schema, bool):
        return "Any"
    
    # Nếu không phải boolean, thử gọi hàm gốc
    try:
        return gradio_client.utils._original_json_schema_to_python_type(schema, defs)
    except (APIInfoParseError, AttributeError, TypeError):
        # Nếu hàm gốc vẫn lỗi, trả về "Any" để app không bị crash
        return "Any"

# 3. Áp dụng bản vá
gradio_client.utils._json_schema_to_python_type = patched_json_schema_to_python_type

print("✅ Đã áp dụng bản vá ULTIMATE cho Gradio.")
print("👉 Bạn có thể chạy Web UI ngay bây giờ!")

In [ ]:
# Code kiểm tra trạng thái an toàn (Không hỏi .device)
import torch

if 'orchestrator' in globals():
    orc = globals()['orchestrator']
    print(f"✅ Orchestrator Class: {type(orc).__name__}")
    
    # Kiểm tra các thành phần bên trong
    components = {
        "Triage (BioMedCLIP)": getattr(orc, 'triage_model', None),
        "VLM (LLaVA)": getattr(orc, 'vlm_model', None),
        "Detector (DINO)": getattr(orc, 'detector_model', None),
        "Segmentor (MedSAM)": getattr(orc, 'segmentor_model', None)
    }
    
    print("\n📦 Trạng thái các model con:")
    for name, model in components.items():
        if model is not None:
            print(f"   - {name}: ✅ Đã load")
        else:
            print(f"   - {name}: ⚠️ Chưa thấy (Có thể load lazy hoặc dùng API)")

else:
    print("❌ Orchestrator chưa được khởi tạo.")

In [ ]:
# # @title 4.2 🏥 TriMedAgent Web Interface (Vietnamese + Visualization)
# # @markdown ✅ Giao diện Tiếng Việt + Tự động vẽ Box/Mask lên ảnh kết quả.

# import gradio as gr
# import nest_asyncio
# import numpy as np
# import torch
# from PIL import Image, ImageDraw, ImageFont
# import matplotlib.pyplot as plt
# import io

# # 1. Fix Asyncio
# nest_asyncio.apply()

# # --- HÀM VẼ ẢNH (VISUALIZATION) ---
# def visualize_result(original_image, result):
#     """
#     Vẽ Bounding Box và Mask lên ảnh gốc.
#     Trả về: Ảnh PIL đã vẽ.
#     """
#     if original_image is None: return None
    
#     # Copy ảnh để vẽ
#     img_draw = original_image.copy().convert("RGBA")
#     draw = ImageDraw.Draw(img_draw)
    
#     # Tạo layer overlay cho Mask (bán trong suốt)
#     mask_overlay = Image.new("RGBA", img_draw.size, (0, 0, 0, 0))
#     mask_draw = ImageDraw.Draw(mask_overlay)
    
#     has_drawing = False

#     # 1. Vẽ Mask (nếu có) - Màu Xanh Dương
#     if hasattr(result, 'masks') and result.masks is not None:
#         for mask in result.masks:
#             # Mask thường là boolean array hoặc tensor
#             if isinstance(mask, torch.Tensor):
#                 mask = mask.cpu().numpy()
            
#             # Resize mask về kích thước ảnh nếu cần
#             if mask.shape[:2] != img_draw.size[::-1]:
#                  # Logic resize phức tạp, ở đây giả định mask khớp size hoặc skip
#                  pass
            
#             # Vẽ mask đơn giản (Bounding box fill) nếu mask phức tạp
#             # Ở đây ta ưu tiên vẽ Box cho nhẹ, nếu muốn vẽ pixel mask cần convert numpy
#             pass 

#     # 2. Vẽ Bounding Box (nếu có) - Màu Đỏ
#     # Ưu tiên box đã kiểm chứng (verified), nếu không có thì lấy raw
#     boxes_to_draw = []
#     if hasattr(result, 'verified_boxes') and result.verified_boxes:
#         boxes_to_draw = result.verified_boxes
#         color = "red"
#         label_prefix = "Phat hien"
#     elif hasattr(result, 'dino_raw_boxes') and result.dino_raw_boxes:
#         boxes_to_draw = result.dino_raw_boxes
#         color = "orange" # Màu cam cho box chưa kiểm chứng
#         label_prefix = "Nghi ngo"

#     for box in boxes_to_draw:
#         has_drawing = True
#         # Box format [x1, y1, x2, y2]
#         x1, y1, x2, y2 = box
        
#         # Vẽ khung
#         draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        
#         # Vẽ nền chữ
#         draw.rectangle([x1, y1, x1+80, y1+20], fill=color)
#         draw.text((x1+5, y1+2), label_prefix, fill="white")

#     # 3. Merge Mask
#     img_draw = Image.alpha_composite(img_draw, mask_overlay)
    
#     if has_drawing:
#         return img_draw.convert("RGB")
#     return None

# # --- LOGIC CHATBOT ---

# def user_message(user_text, history, image_state):
#     return "", history + [[user_text, None]], image_state

# def bot_response(history, image_state, model_selector):
#     user_text = history[-1][0]
    
#     if 'orchestrator' not in globals():
#         history[-1][1] = "⚠️ Lỗi: Orchestrator chưa khởi tạo (Chạy Cell 3.1 đi bạn!)"
#         yield history
#         return

#     try:
#         # Gọi Orchestrator
#         response_text, result = orchestrator.chat(user_text, image_state)
        
#         # 1. Hiển thị text trả lời (Tiếng Việt từ LLaVA)
#         history[-1][1] = response_text
#         yield history
        
#         # 2. Kiểm tra và Vẽ hình (Nếu có detection)
#         # Nếu orchestrator tìm thấy box, ta vẽ và gửi thêm 1 tin nhắn ảnh
#         annotated_img = visualize_result(image_state, result)
        
#         if annotated_img:
#             # Lưu ảnh vào RAM để Gradio hiển thị
#             import tempfile
#             with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tfile:
#                 annotated_img.save(tfile.name)
#                 img_path = tfile.name
            
#             # Gửi ảnh vào chat
#             history.append((None, (img_path, "Kết quả khoanh vùng")))
#             yield history

#     except Exception as e:
#         history[-1][1] = f"❌ Lỗi: {str(e)}"
#         print(f"Error: {e}")
#         yield history

# def clear_session():
#     if 'orchestrator' in globals(): orchestrator.clear_session()
#     return None, [], None

# def handle_image_upload(image, history):
#     if image is not None:
#         return image, history + [[None, "📸 Đã nhận ảnh. Bác sĩ cần tôi giúp gì?"]]
#     return None, history

# # --- GIAO DIỆN (VIETNAMESE) ---

# with gr.Blocks(title="TriMedAgent V2 (VN)", theme=gr.themes.Soft()) as demo:
#     gr.Markdown("# 🏥 TriMedAgent V2 - Trợ Lý Chẩn Đoán AI")
#     gr.Markdown("Hệ thống tích hợp: **LLaVA-Med** (Tư duy) | **MedSAM** (Khoanh vùng) | **BiomedCLIP** (Phân loại)")
    
#     image_state = gr.State(None)
    
#     with gr.Row():
#         # Cột Chat
#         with gr.Column(scale=7):
#             chatbot = gr.Chatbot(
#                 height=550, 
#                 label="Hội chẩn trực tuyến", 
#                 bubble_full_width=False,
#                 avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/3774/3774299.png")
#             )
#             with gr.Row():
#                 txt_input = gr.Textbox(
#                     scale=8, 
#                     placeholder="Nhập câu hỏi (VD: 'Tìm khối u phổi', 'Bệnh nhân bị gì?')...", 
#                     container=False, autofocus=True
#                 )
#                 btn_send = gr.Button("Gửi", variant="primary", scale=1)

#         # Cột Công cụ
#         with gr.Column(scale=3):
#             img_input = gr.Image(type="pil", label="Tải ảnh X-quang/CT/MRI", height=250)
            
#             gr.Markdown("### ⚡ Chọn nhanh")
#             with gr.Row():
#                 btn_desc = gr.Button("📝 Mô tả ảnh")
#                 btn_detect = gr.Button("🔍 Tìm & Khoanh vùng")
            
#             with gr.Accordion("⚙️ Cài đặt Model", open=False):
#                 model_dd = gr.Dropdown(["TriMed-Mistral"], label="Model", value="TriMed-Mistral")
            
#             btn_clear = gr.Button("🗑️ Xóa & Làm mới")

#     # --- SỰ KIỆN ---
#     img_input.upload(handle_image_upload, [img_input, chatbot], [image_state, chatbot])
    
#     # Xử lý Chat
#     txt_input.submit(user_message, [txt_input, chatbot, image_state], [txt_input, chatbot, image_state]).then(
#         bot_response, [chatbot, image_state, model_dd], [chatbot]
#     )
#     btn_send.click(user_message, [txt_input, chatbot, image_state], [txt_input, chatbot, image_state]).then(
#         bot_response, [chatbot, image_state, model_dd], [chatbot]
#     )

#     # Nút nhanh
#     btn_desc.click(lambda h, i: ("", h + [["Mô tả chi tiết ảnh này.", None]], i), [chatbot, image_state], [txt_input, chatbot, image_state]).then(
#         bot_response, [chatbot, image_state, model_dd], [chatbot]
#     )
#     # Lưu ý: Prompt này kích hoạt DINO trong Orchestrator
#     btn_detect.click(lambda h, i: ("", h + [["Hãy tìm và khoanh vùng tổn thương (find and segment).", None]], i), [chatbot, image_state], [txt_input, chatbot, image_state]).then(
#         bot_response, [chatbot, image_state, model_dd], [chatbot]
#     )

#     btn_clear.click(clear_session, None, [image_state, chatbot, img_input])

# print("🚀 Đang khởi động TriMedAgent Tiếng Việt...")
# demo.queue().launch(share=True, debug=True, show_api=False, allowed_paths=["/content"])

---
## 📋 Summary

### ✅ What's New in V2:

| Feature | V1 | V2 |
|---------|----|----|
| Single-turn Q&A | ✅ | ✅ |
| **Multi-turn Memory** | ❌ | ✅ |
| **Context Awareness** | ❌ | ✅ |
| **Session Management** | ❌ | ✅ |
| Detection Pipeline | ✅ | ✅ |
| Segmentation | ✅ | ✅ |

### 🎯 Key Methods:

```python
# Multi-turn chat
response, result = orchestrator.chat("Your question", image)

# Follow-up (no image needed)
response2, result2 = orchestrator.chat("Follow-up question")

# View history
history = orchestrator.get_chat_history()

# Clear session
orchestrator.clear_session()
```

### ⚠️ Troubleshooting Gradio Issues

| Lỗi | Nguyên nhân | Fix |
|-----|-------------|-----|
| `loop_factory` TypeError | Xung đột asyncio | Chạy `!pip install nest_asyncio` và `nest_asyncio.apply()` |
| `type='tuples'` warning | Gradio deprecation | Dùng `type='messages'` với format `{"role": "user", "content": "..."}` |
| `bubble_full_width` warning | Parameter removed | Xóa parameter này |
| Server không khởi động | Port conflict | Restart runtime hoặc thêm `server_port=7861` |

### 🚀 Next Steps:

- **Add RAG**: See `demo_trimedagent_rag.ipynb` for knowledge retrieval
- **Production**: Deploy with HTTP workers for scalability
- **Fine-tuning**: Train on domain-specific medical data

---

📚 **Full Documentation**: See `docs/V2_MULTI_TURN_RAG_GUIDE.md`